# Spike de exploracion — `contract_multifile` (T-57)

> **Paso 4 del flujo** (`notebook_writer`). Prototipo **visual** de las historias `HU-01 … HU-12` de `definition.md`, para el **gate humano del paso 5**.

## Que se prototipa
El cambio de forma del Contrato de Datos: de `Contract.columnas: list[Columna]` (un solo archivo) a **`Contract.archivos: list[ArchivoContrato]`** (uno o varios), tal como fija **D-18 (🔒 resuelta)**:

```yaml
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
```

El spike **reutiliza sin reescribir** el nucleo ya verificado en `config_contract` (T-43) — se importa literalmente de `app/src/zeroleak/config/contract.py`: `Columna`, el enum cerrado `TipoDato` de 6 tipos (D-23b), la deteccion de duplicados por cadena exacta (D-23e), el fail-fast de un solo error (D-23c) y las excepciones `ContractParseError` / `ContractSchemaError`. Lo nuevo es **un nivel mas de anidamiento** que envuelve ese nucleo.

## Reglas que este spike respeta
- **🔒 C-01 (Datos en Boveda):** solo **YAMLs sinteticos** escritos aqui mismo (`clientes.csv`, `ventas.csv`, `test_id`…), materializados en un directorio **temporal**. Ningun dato real de cliente; nada se lee de `clients/*/data/`. Verificado en la Celda 16.
- **Frontera D-21:** se valida el contrato **contra su propio esquema**; **no** se abre, lee ni compara ningun CSV de bronze. Verificado en la Celda 14.
- **Core-only (D-23d):** sin fachada CLI.
- **Spike, no producto (Opcion A):** explora, no es fuente de verdad. **No se modifica** `app/src/zeroleak/config/contract.py` ni sus tests: el codigo de produccion lo reescriben la spec (paso 6) y el bucle TDD (paso 10). Este notebook queda como documentacion de referencia.

## Lo que este spike debe iluminar para el gate (sin decidirlo)
La `definition.md` delego dos puntos a la spec. El spike **muestra la evidencia** de cada opcion; **decide el humano**:

- **(a) Como identifica el mensaje de error al archivo:** por indice (`archivo[1]`), por `nombre` (`archivo 'ventas.csv'`) o **ambos**. Ver **Celda 7** (el caso feo: si lo que falta es justamente el `nombre`, no hay nombre con que identificar) y la comparativa de la **Celda 10**.
- **(b) Duplicado de `nombre` de archivo por cadena exacta (supuesto asumido, D-23e) o normalizada** (mayusculas/espacios). Ver **Celda 8**.

## ⚠️ Estado de ejecucion
El agente que escribio este notebook **no dispone de entorno de ejecucion**, por lo que las celdas estan **sin salidas**. Antes del gate del paso 5 debe ejecutarse de arriba a abajo (el proyecto exige Python 3.13):

```powershell
py -3.13 -m jupyter nbconvert --to notebook --execute --inplace 610_features/contract_multifile/contract_multifile.ipynb
```

Sin salidas visibles el notebook **no sirve** para el gate.

## Celda 1 — Nucleo VIGENTE importado (reuso, no reescritura)

Traza → base de todas las HU; evidencia directa de **HU-02** (no duplicar logica) y **HU-11** (migracion sin perder cobertura).

Importamos el modulo real `zeroleak.config.contract` **tal como esta hoy en `main`**. Nos sirve para dos cosas:
1. **Reutilizar** `Columna`, `TipoDato`, `ContractParseError`, `ContractSchemaError` en el prototipo (no se redefinen).
2. **Contrastar** el comportamiento de HOY contra la propuesta (Celdas 5, 9 y 12: mensajes ambiguos y el bug T-56).

In [1]:
import sys
import tempfile
from pathlib import Path

import pandas as pd
import yaml
from pydantic import BaseModel, ValidationError, field_validator

pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', 220)


def _raiz_repo(inicio: Path) -> Path:
    """Busca hacia arriba la raiz del repo (la que contiene app/src/zeroleak)."""
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'app' / 'src' / 'zeroleak').is_dir():
            return candidato
    raise RuntimeError(f'No se encontro la raiz del repo desde {inicio}')


RAIZ = _raiz_repo(Path.cwd().resolve())
sys.path.insert(0, str(RAIZ / 'app' / 'src'))

# --- Nucleo VIGENTE: se importa, NO se redefine ni se modifica ---------------
from zeroleak.config.contract import (  # noqa: E402
    _MENSAJE_YAML_INVALIDO,
    Columna,
    ContractParseError,
    ContractSchemaError,
    TipoDato,
)
from zeroleak.config.contract import Contract as ContractVigente  # noqa: E402
from zeroleak.config.contract import _mensaje_esquema as _mensaje_esquema_vigente  # noqa: E402
from zeroleak.config.contract import load_contract as load_contract_vigente  # noqa: E402

print('Raiz del repo :', RAIZ)
print('Modulo vigente:', 'zeroleak.config.contract (importado tal cual, sin tocar)')
print()
print('Piezas REUTILIZADAS por el prototipo:')
print('  TipoDato (enum cerrado, D-23b) :', [t.value for t in TipoDato])
print('  Columna.model_fields           :', list(Columna.model_fields))
print('  Excepciones                    :', ContractParseError.__name__, '/', ContractSchemaError.__name__)
print()
print('Pieza que CAMBIA de forma (esta feature):')
print('  HOY  -> Contract.model_fields  :', list(ContractVigente.model_fields))
print("  MANANA -> Contract.model_fields:", ['archivos'], '(cada uno con nombre + columnas; D-18)')

Raiz del repo : C:\Users\USUARIO\Documents\TripleS\ZeroLeak_Application
Modulo vigente: zeroleak.config.contract (importado tal cual, sin tocar)

Piezas REUTILIZADAS por el prototipo:
  TipoDato (enum cerrado, D-23b) : ['string', 'integer', 'float', 'date', 'datetime', 'boolean']
  Columna.model_fields           : ['nombre', 'tipo', 'nulable', 'llave']
  Excepciones                    : ContractParseError / ContractSchemaError

Pieza que CAMBIA de forma (esta feature):
  HOY  -> Contract.model_fields  : ['columnas']
  MANANA -> Contract.model_fields: ['archivos'] (cada uno con nombre + columnas; D-18)


## Celda 2 — Datos sinteticos (matrices de mentiras)

Traza → **HU-12** (fixtures 100% sinteticos, sin PII, versionables).

Todos los contratos de prueba —validos y deliberadamente rotos— se escriben aqui y se materializan en un directorio **temporal** del sistema. Ninguno proviene de un cliente real; ninguna ruta apunta a `clients/*/data/`. Los archivos que nombran (`clientes.csv`, `ventas.csv`) son **cadenas declarativas**: el spike nunca los busca ni los abre (frontera D-21).

In [2]:
FIXTURES = Path(tempfile.mkdtemp(prefix='contract_multifile_spike_'))

# Cada entrada: nombre -> (proposito/traza, contenido YAML sintetico)
fixtures: dict[str, tuple[str, str]] = {}

# --- Camino feliz: 2 archivos (forma D-18) -----------------------------------
fixtures['multi_valido.yaml'] = ('HU-01 camino feliz multi-archivo', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
        - {nombre: correo, tipo: string, nulable: true, llave: false}
        - {nombre: fecha_alta, tipo: date, nulable: false, llave: false}
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: false}
        - {nombre: monto, tipo: float, nulable: true, llave: false}
        - {nombre: vendida_en, tipo: datetime, nulable: false, llave: false}
        - {nombre: anulada, tipo: boolean, nulable: false, llave: false}
''')

# --- Un solo archivo: el caso general con len(archivos) == 1 ------------------
fixtures['un_archivo.yaml'] = ('HU-02 un solo archivo (forma NUEVA)', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
        - {nombre: correo, tipo: string, nulable: true, llave: false}
        - {nombre: fecha_alta, tipo: date, nulable: false, llave: false}
''')

# Mismo contenido en la forma VIEJA, para comparar equivalencia (HU-02/HU-11)
fixtures['un_archivo_esquema_viejo.yaml'] = ('HU-02/HU-11 mismo contrato en la forma VIEJA', '''
contract_data:
  columnas:
    - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - {nombre: correo, tipo: string, nulable: true, llave: false}
    - {nombre: fecha_alta, tipo: date, nulable: false, llave: false}
''')

# --- HU-03: archivos vacia / nula / ausente ----------------------------------
fixtures['archivos_vacia.yaml'] = ('HU-03 archivos: []', '''
contract_data:
  archivos: []
''')

fixtures['archivos_nula.yaml'] = ('HU-03 archivos: (nulo)', '''
contract_data:
  archivos:
''')

fixtures['archivos_ausente_esquema_viejo.yaml'] = ('HU-03 clave archivos ausente (contrato del esquema viejo)', '''
contract_data:
  columnas:
    - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
''')

# --- HU-04: archivo sin nombre / columnas vacias / columnas ausentes ---------
fixtures['archivo_sin_nombre.yaml'] = ('HU-04 el 2do archivo no declara nombre (CASO FEO)', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
''')

fixtures['archivo_columnas_vacias.yaml'] = ('HU-04 el 2do archivo tiene columnas: []', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas: []
''')

fixtures['archivo_columnas_ausentes.yaml'] = ('HU-04 el 2do archivo no declara columnas', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
''')

# --- HU-05: nombre de archivo duplicado --------------------------------------
fixtures['archivos_duplicados.yaml'] = ('HU-05 dos archivos ventas.csv (cadena exacta)', '''
contract_data:
  archivos:
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: monto, tipo: float, nulable: true, llave: false}
''')

fixtures['archivos_duplicados_casing.yaml'] = ('HU-05/(b) Ventas.csv vs ventas.csv (difieren solo en mayusculas)', '''
contract_data:
  archivos:
    - nombre: Ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: monto, tipo: float, nulable: true, llave: false}
''')

# --- HU-06: validaciones de columna dentro del 2do archivo -------------------
fixtures['columna_campo_faltante.yaml'] = ('HU-06 ventas.csv: la columna monto no declara tipo', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
        - {nombre: monto, nulable: true, llave: false}
''')

fixtures['columna_tipo_invalido.yaml'] = ('HU-06 ventas.csv: tipo numero_magico fuera del enum de 6', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
        - {nombre: monto, tipo: numero_magico, nulable: true, llave: false}
''')

fixtures['columnas_duplicadas.yaml'] = ('HU-06 ventas.csv: columna venta_id duplicada', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
        - {nombre: venta_id, tipo: string, nulable: true, llave: false}
''')

# --- HU-07: sintaxis YAML rota ----------------------------------------------
fixtures['yaml_roto.yaml'] = ('HU-07 indentacion invalida (no parsea)', '''
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - nombre: cliente_id
          tipo: integer
         nulable: false
           llave: true
''')

# --- HU-08 (T-56): contract_data presente pero nulo --------------------------
fixtures['contract_data_nulo.yaml'] = ('HU-08/T-56 contract_data: sin nada debajo', '''
contract_data:
''')

fixtures['yaml_totalmente_vacio.yaml'] = ('HU-08 archivo sin contenido (solo comentario)', '''
# contrato aun sin escribir
''')

fixtures['contract_data_no_mapa.yaml'] = ('HU-08 contract_data con un valor que no es un mapa', '''
contract_data: pendiente_de_completar
''')

for nombre, (_proposito, contenido) in fixtures.items():
    (FIXTURES / nombre).write_text(contenido, encoding='utf-8')

print('Fixtures sinteticos escritos en un directorio TEMPORAL:')
print(' ', FIXTURES)
print()
indice_fixtures = pd.DataFrame(
    [{'fixture': n, 'proposito / traza': p} for n, (p, _c) in fixtures.items()]
)
indice_fixtures

Fixtures sinteticos escritos en un directorio TEMPORAL:
  C:\Users\USUARIO\AppData\Local\Temp\contract_multifile_spike_na5n32l9



,fixture,proposito / traza
0,multi_valido.yaml,HU-01 camino feliz multi-archivo
1,un_archivo.yaml,HU-02 un solo archivo (forma NUEVA)
2,un_archivo_esquema_viejo.yaml,HU-02/HU-11 mismo contrato en la forma VIEJA
3,archivos_vacia.yaml,HU-03 archivos: []
4,archivos_nula.yaml,HU-03 archivos: (nulo)
5,archivos_ausente_esquema_viejo.yaml,HU-03 clave archivos ausente (contrato del esquema viejo)
6,archivo_sin_nombre.yaml,HU-04 el 2do archivo no declara nombre (CASO FEO)
7,archivo_columnas_vacias.yaml,HU-04 el 2do archivo tiene columnas: []
8,archivo_columnas_ausentes.yaml,HU-04 el 2do archivo no declara columnas
9,archivos_duplicados.yaml,HU-05 dos archivos ventas.csv (cadena exacta)


## Celda 3 — El prototipo: `ArchivoContrato` + `Contract.archivos` + `load_contract`

Traza → base de **HU-01 … HU-08**.

Enfoque explorado (el que se propone a la spec):

| Pieza | De donde sale |
|---|---|
| `Columna`, `TipoDato`, `ContractParseError`, `ContractSchemaError` | **importados sin tocar** del modulo vigente |
| Validadores de columna (`no vacias`, `sin duplicados`) | **mismo codigo, reubicado** un nivel: de `Contract` a `ArchivoContrato` |
| `ArchivoContrato = {nombre: str, columnas: list[Columna]}` | **nuevo** |
| `ContractPropuesto = {archivos: list[ArchivoContrato]}` | evolucion de `Contract` |
| `load_contract_propuesto(path)` | misma firma y misma secuencia (leer → parsear → validar) |

Dos cambios de fondo respecto de hoy:
- **Extraccion defensiva de la raiz** (`contract_data` debe ser un mapa) → resuelve **T-56** sin `AttributeError` (Celda 12).
- **`_mensaje_esquema_propuesto(exc, archivos_crudos, estilo)`**: traduce el **primer** error de Pydantic (fail-fast, D-23c) **localizando el archivo**. El parametro `estilo` (`'indice'` / `'nombre'` / `'ambos'`) existe **solo en el spike**, para poner la pregunta abierta (a) delante del humano en la Celda 10. La spec fija **uno**.

> Nota tecnica: el nombre del archivo **no viene en el error de Pydantic** cuando el defecto esta en una columna (`loc = ('archivos', 1, 'columnas', 2, 'tipo')`). Se recupera del **YAML crudo** por indice. Este es el hallazgo que condiciona la respuesta a (a).

In [3]:
import inspect

# --- Pregunta abierta (b): politica de duplicados de NOMBRE DE ARCHIVO -------
# Supuesto asumido en definition.md (consistente con D-23e): cadena exacta.
# Este flag existe SOLO en el spike, para contrastar ambas opciones (Celda 8).
MODO_DUPLICADOS_NORMALIZADO = False


def _clave_duplicado(nombre: str, normalizar: bool) -> str:
    return nombre.strip().lower() if normalizar else nombre


def _duplicados(nombres, normalizar: bool = False) -> list[str]:
    """Nombres repetidos, en orden de aparicion (misma logica que el vigente)."""
    vistos: set[str] = set()
    ya_reportados: set[str] = set()
    duplicados: list[str] = []
    for nombre in nombres:
        clave = _clave_duplicado(nombre, normalizar)
        if clave in vistos and clave not in ya_reportados:
            duplicados.append(nombre)
            ya_reportados.add(clave)
        vistos.add(clave)
    return duplicados


# --- NUEVO: un archivo declarado en el contrato (D-18) -----------------------
class ArchivoContrato(BaseModel):
    nombre: str
    columnas: list[Columna]          # <- Columna es la IMPORTADA, sin cambios

    # Los dos validadores de abajo son el MISMO codigo que hoy vive en
    # `Contract` (contract.py:60-84): aqui solo CAMBIAN DE CASA, para validar
    # las columnas DE ESTE ARCHIVO. Misma logica y mismo mensaje (D-23e).
    @field_validator('columnas')
    @classmethod
    def _columnas_no_vacias(cls, valor):
        if not valor:
            raise ValueError('la lista de columnas no puede estar vacía')
        return valor

    @field_validator('columnas')
    @classmethod
    def _columnas_sin_duplicados(cls, valor):
        duplicados = _duplicados([c.nombre for c in valor])   # cadena exacta (D-23e)
        if duplicados:
            nombres = ', '.join(f"'{n}'" for n in duplicados)
            raise ValueError(f'nombre de columna duplicado: {nombres}')
        return valor


# --- Contract: `columnas` -> `archivos` (el cambio de esta feature) ----------
class ContractPropuesto(BaseModel):
    archivos: list[ArchivoContrato]

    @field_validator('archivos')
    @classmethod
    def _archivos_no_vacia(cls, valor):
        if not valor:
            raise ValueError('la lista de archivos no puede estar vacía')
        return valor

    @field_validator('archivos')
    @classmethod
    def _archivos_sin_duplicados(cls, valor):
        duplicados = _duplicados([a.nombre for a in valor], MODO_DUPLICADOS_NORMALIZADO)
        if duplicados:
            nombres = ', '.join(f"'{n}'" for n in duplicados)
            raise ValueError(f'nombre de archivo duplicado: {nombres}')
        return valor


# --- Pregunta abierta (a): como se nombra al archivo en el mensaje -----------
def _localizador_archivo(indice, archivos_crudos, estilo='ambos') -> str:
    """Devuelve el texto con que el mensaje identifica al archivo.

    El nombre NO viene en el error de Pydantic cuando el defecto esta en una
    columna: se recupera del YAML crudo por indice. Si el archivo no declara
    `nombre` (HU-04, caso feo), no hay nombre y solo queda el indice.
    """
    nombre = None
    if isinstance(indice, int) and 0 <= indice < len(archivos_crudos):
        item = archivos_crudos[indice]
        if isinstance(item, dict) and isinstance(item.get('nombre'), str) and item['nombre']:
            nombre = item['nombre']
    if estilo == 'indice' or nombre is None:
        return f'archivo[{indice}]'
    if estilo == 'nombre':
        return f"archivo '{nombre}'"
    return f"archivo[{indice}] '{nombre}'"


def _mensaje_esquema_propuesto(exc: ValidationError, archivos_crudos, estilo='ambos') -> str:
    """Traduce el PRIMER error de Pydantic localizando el archivo (D-23c).

    Formas de `loc` que hay que desarmar ahora:
      ('archivos',)                              -> lista vacia / duplicados
      ('archivos', i, campo)                     -> falta `nombre`/`columnas`, o columnas vacia/duplicada
      ('archivos', i, 'columnas', j, campo)      -> error de columna dentro del archivo i
    """
    primer_error = exc.errors()[0]
    loc = primer_error.get('loc', ())
    tipo_error = primer_error.get('type', '')
    msg = primer_error.get('msg', '')
    limpio = msg.removeprefix('Value error, ')

    # Nivel raiz: la propia lista `archivos`
    if loc == ('archivos',):
        return limpio if tipo_error == 'value_error' else f"campo 'archivos': {msg}"

    indice_archivo = loc[1] if len(loc) > 1 else '?'
    donde = _localizador_archivo(indice_archivo, archivos_crudos, estilo)

    # Nivel archivo: ('archivos', i, campo)
    if len(loc) == 3:
        campo = loc[2]
        if tipo_error == 'missing':
            return f"{donde}: falta el campo requerido '{campo}' ({msg})"
        if tipo_error == 'value_error':
            return f'{donde}: {limpio}'
        return f"{donde}, campo '{campo}': {msg}"

    # Nivel columna: ('archivos', i, 'columnas', j, campo)
    if len(loc) >= 5:
        indice_columna = loc[3]
        campo = loc[-1]
        if tipo_error == 'missing':
            return (
                f'{donde}, columna de índice {indice_columna}: falta el campo '
                f"requerido '{campo}' ({msg})"
            )
        if tipo_error == 'enum':
            valor_invalido = primer_error.get('input', '?')
            return (
                f"{donde}, columna de índice {indice_columna}, campo '{campo}': "
                f"valor '{valor_invalido}' inválido ({msg})"
            )
        return f"{donde}, columna de índice {indice_columna}, campo '{campo}': {msg}"

    return f'{donde}: {msg}'


# --- Core propuesto: misma firma, misma secuencia leer -> parsear -> validar --
def load_contract_propuesto(path, estilo='ambos') -> ContractPropuesto:
    """Prototipo de `load_contract` multi-archivo.

    (`estilo` es un parametro SOLO del spike, para la pregunta abierta (a);
    la firma real sigue siendo `load_contract(path) -> Contract`.)
    """
    with open(path, 'r', encoding='utf-8') as fh:
        try:
            crudo = yaml.safe_load(fh)
        except yaml.YAMLError as exc:
            raise ContractParseError(f'{_MENSAJE_YAML_INVALIDO}: {exc}') from exc

    # T-56: la clave raiz puede EXISTIR con valor nulo. Se comprueba el tipo en
    # vez de encadenar `.get(...).get(...)`  ->  nunca un AttributeError.
    raiz = crudo if isinstance(crudo, dict) else {}
    cuerpo = raiz.get('contract_data')
    if not isinstance(cuerpo, dict):
        raise ContractSchemaError(
            "la clave raíz 'contract_data' debe contener un mapa con la lista "
            f"'archivos' (se encontró: {cuerpo!r})"
        )

    archivos_crudos = cuerpo.get('archivos') or []      # ausente o nula -> []
    if not isinstance(archivos_crudos, list):
        raise ContractSchemaError(
            f"'archivos' debe ser una lista (se encontró: {archivos_crudos!r})"
        )

    try:
        return ContractPropuesto.model_validate({'archivos': archivos_crudos})
    except ValidationError as exc:
        raise ContractSchemaError(
            _mensaje_esquema_propuesto(exc, archivos_crudos, estilo)
        ) from exc


print('Prototipo definido (core-only, sin CLI; D-23d):')
print('  ArchivoContrato.model_fields  :', list(ArchivoContrato.model_fields))
print('  ContractPropuesto.model_fields:', list(ContractPropuesto.model_fields))
print('  Firma del core                : load_contract_propuesto',
      inspect.signature(load_contract_propuesto))
print()
print('Reutilizado sin reescribir  :', 'Columna, TipoDato, ContractParseError, ContractSchemaError')
print('Reubicado (mismo codigo)    :', '_columnas_no_vacias, _columnas_sin_duplicados -> ArchivoContrato')
print('Nuevo                       :', '_archivos_no_vacia, _archivos_sin_duplicados, localizacion por archivo')

Prototipo definido (core-only, sin CLI; D-23d):
  ArchivoContrato.model_fields  : ['nombre', 'columnas']
  ContractPropuesto.model_fields: ['archivos']
  Firma del core                : load_contract_propuesto (path, estilo='ambos') -> __main__.ContractPropuesto

Reutilizado sin reescribir  : Columna, TipoDato, ContractParseError, ContractSchemaError
Reubicado (mismo codigo)    : _columnas_no_vacias, _columnas_sin_duplicados -> ArchivoContrato
Nuevo                       : _archivos_no_vacia, _archivos_sin_duplicados, localizacion por archivo


## Celda 4 — HU-01: camino feliz multi-archivo → `Contract.archivos` fiel a lo declarado

Traza → **HU-01**. Cargamos un YAML con **dos** archivos y mostramos en una **tabla** que cada archivo conserva su `nombre` y su lista de columnas (nombre, tipo, nulable, llave) **en el orden declarado**.

In [4]:
contrato = load_contract_propuesto(FIXTURES / 'multi_valido.yaml')

print('Tipo del objeto devuelto :', type(contrato).__name__)
print('Numero de archivos       :', len(contrato.archivos))
print('Nombres, en orden        :', [a.nombre for a in contrato.archivos])
print()

tabla_contrato = pd.DataFrame(
    [
        {
            'archivo': archivo.nombre,
            'pos_archivo': i,
            'pos_columna': j,
            'columna': columna.nombre,
            'tipo': columna.tipo.value,
            'nulable': columna.nulable,
            'llave': columna.llave,
        }
        for i, archivo in enumerate(contrato.archivos)
        for j, columna in enumerate(archivo.columnas)
    ]
)
tabla_contrato

Tipo del objeto devuelto : ContractPropuesto
Numero de archivos       : 2
Nombres, en orden        : ['clientes.csv', 'ventas.csv']



,archivo,pos_archivo,pos_columna,columna,tipo,nulable,llave
0,clientes.csv,0,0,cliente_id,integer,False,True
1,clientes.csv,0,1,correo,string,True,False
2,clientes.csv,0,2,fecha_alta,date,False,False
3,ventas.csv,1,0,venta_id,integer,False,True
4,ventas.csv,1,1,cliente_id,integer,False,False
5,ventas.csv,1,2,monto,float,True,False
6,ventas.csv,1,3,vendida_en,datetime,False,False
7,ventas.csv,1,4,anulada,boolean,False,False


## Celda 5 — HU-02: un solo archivo es el **caso general**, no una rama especial

Traza → **HU-02**. Dos evidencias:
1. Un YAML con un unico archivo carga por **el mismo camino** y da `len(archivos) == 1` (cero ramas `if`).
2. Las columnas obtenidas son **equivalentes** a las que producia la forma vieja (`Contract.columnas`) sobre el mismo contenido: se compara `[c.model_dump() for c in viejo.columnas]` contra `[c.model_dump() for c in nuevo.archivos[0].columnas]` usando el **mismo** modelo `Columna` importado.

In [5]:
nuevo = load_contract_propuesto(FIXTURES / 'un_archivo.yaml')
viejo = load_contract_vigente(FIXTURES / 'un_archivo_esquema_viejo.yaml')

print('--- Forma NUEVA con un solo archivo (mismo camino, sin rama especial) ---')
print('len(archivos)      :', len(nuevo.archivos))
print('archivos[0].nombre :', nuevo.archivos[0].nombre)
print('columnas           :', [c.nombre for c in nuevo.archivos[0].columnas])
print()

columnas_viejas = [c.model_dump() for c in viejo.columnas]
columnas_nuevas = [c.model_dump() for c in nuevo.archivos[0].columnas]

print('--- Equivalencia con la forma VIEJA (mismo modelo `Columna` importado) ---')
print('Contract vigente     -> columnas           :', [c.nombre for c in viejo.columnas])
print('Contract propuesto   -> archivos[0].columnas:', [c.nombre for c in nuevo.archivos[0].columnas])
print()
print('¿Contenido identico? ->', columnas_viejas == columnas_nuevas)
assert len(nuevo.archivos) == 1
assert columnas_viejas == columnas_nuevas
print()
print('OK: un archivo es el caso general (len(archivos) == 1); no se pierde nada')
print('    respecto de la forma vieja y no hace falta ninguna rama `if`.')

pd.DataFrame(
    [
        {'forma': 'VIEJA  Contract.columnas', **c}
        for c in columnas_viejas
    ]
    + [
        {'forma': 'NUEVA  archivos[0].columnas', **c}
        for c in columnas_nuevas
    ]
)

--- Forma NUEVA con un solo archivo (mismo camino, sin rama especial) ---


len(archivos)      : 1
archivos[0].nombre : clientes.csv
columnas           : ['cliente_id', 'correo', 'fecha_alta']

--- Equivalencia con la forma VIEJA (mismo modelo `Columna` importado) ---
Contract vigente     -> columnas           : ['cliente_id', 'correo', 'fecha_alta']
Contract propuesto   -> archivos[0].columnas: ['cliente_id', 'correo', 'fecha_alta']

¿Contenido identico? -> True

OK: un archivo es el caso general (len(archivos) == 1); no se pierde nada
    respecto de la forma vieja y no hace falta ninguna rama `if`.


,forma,nombre,tipo,nulable,llave
0,VIEJA Contract.columnas,cliente_id,TipoDato.INTEGER,False,True
1,VIEJA Contract.columnas,correo,TipoDato.STRING,True,False
2,VIEJA Contract.columnas,fecha_alta,TipoDato.DATE,False,False
3,NUEVA archivos[0].columnas,cliente_id,TipoDato.INTEGER,False,True
4,NUEVA archivos[0].columnas,correo,TipoDato.STRING,True,False
5,NUEVA archivos[0].columnas,fecha_alta,TipoDato.DATE,False,False


## Celda 6 — HU-03: `archivos` vacia o ausente → error de esquema claro

Traza → **HU-03**. Tres variantes del mismo hueco, todas deben dar `ContractSchemaError` y **ningun objeto**:
- `archivos: []` (vacia),
- `archivos:` con valor nulo,
- clave `archivos` **ausente** — que es justo el caso de un contrato escrito en el **esquema viejo** (`contract_data.columnas`). Ese fixture es el mas realista durante la migracion.

In [6]:
def probar(caso: str, estilo: str = 'ambos') -> tuple[str, str]:
    """Carga un fixture y devuelve (resultado, mensaje) sin cortar el notebook.

    Captura tambien `Exception` a proposito: si algo se escapa que no sea
    ContractParseError/ContractSchemaError, es una fuga y queda a la vista.
    """
    try:
        obj = load_contract_propuesto(FIXTURES / caso, estilo=estilo)
        return 'OK -> Contract', f'{len(obj.archivos)} archivo(s): ' + ', '.join(
            a.nombre for a in obj.archivos
        )
    except ContractParseError as exc:
        return 'ContractParseError', str(exc).splitlines()[0]
    except ContractSchemaError as exc:
        return 'ContractSchemaError', str(exc)
    except Exception as exc:                     # noqa: BLE001  (a proposito)
        return f'!! {type(exc).__name__} NO CONTROLADA', str(exc)


casos_hu03 = [
    'archivos_vacia.yaml',
    'archivos_nula.yaml',
    'archivos_ausente_esquema_viejo.yaml',
]

filas = []
for caso in casos_hu03:
    resultado, mensaje = probar(caso)
    filas.append({'fixture': caso, 'resultado': resultado, 'mensaje': mensaje})
    print(f'=== {caso} ===')
    print(f'  {resultado}: {mensaje}')
    print('  objeto devuelto: None (no se construye un contrato a medias)')
    print()

assert all(f['resultado'] == 'ContractSchemaError' for f in filas)
print('OK: vacia, nula y ausente -> el mismo ContractSchemaError claro; ningun objeto.')
pd.DataFrame(filas)

=== archivos_vacia.yaml ===
  ContractSchemaError: la lista de archivos no puede estar vacía
  objeto devuelto: None (no se construye un contrato a medias)

=== archivos_nula.yaml ===
  ContractSchemaError: la lista de archivos no puede estar vacía
  objeto devuelto: None (no se construye un contrato a medias)

=== archivos_ausente_esquema_viejo.yaml ===
  ContractSchemaError: la lista de archivos no puede estar vacía
  objeto devuelto: None (no se construye un contrato a medias)

OK: vacia, nula y ausente -> el mismo ContractSchemaError claro; ningun objeto.


,fixture,resultado,mensaje
0,archivos_vacia.yaml,ContractSchemaError,la lista de archivos no puede estar vacía
1,archivos_nula.yaml,ContractSchemaError,la lista de archivos no puede estar vacía
2,archivos_ausente_esquema_viejo.yaml,ContractSchemaError,la lista de archivos no puede estar vacía


## Celda 7 — HU-04: archivo sin `nombre` o con `columnas` vacias/ausentes → error que **identifica el archivo**

Traza → **HU-04**. Tres fixtures, todos con **dos** archivos donde el defectuoso es el **segundo**.

### 🔍 El caso feo (lo que el humano necesita ver)
Cuando lo que falta es justamente el **`nombre`**, **no hay nombre con el cual identificar el archivo**. Solo queda:
- su **posicion** en la lista (`archivo[1]`), o
- describirlo por su contenido (p. ej. sus columnas), que es mas ruidoso y tampoco es unico.

Esto acota la pregunta abierta **(a)**: una politica de "identificar **solo** por `nombre`" **no cubre este caso**. Abajo se ve el mensaje realmente producido en los tres estilos.

In [7]:
casos_hu04 = [
    'archivo_sin_nombre.yaml',
    'archivo_columnas_vacias.yaml',
    'archivo_columnas_ausentes.yaml',
]

filas = []
for caso in casos_hu04:
    resultado, mensaje = probar(caso)
    filas.append({'fixture': caso, 'resultado': resultado, 'mensaje': mensaje})
    print(f'=== {caso} ===')
    print(f'  {resultado}: {mensaje}')
    print()

assert all(f['resultado'] == 'ContractSchemaError' for f in filas)
print('OK: los tres huecos de nivel archivo -> ContractSchemaError que ubica el archivo.')
print()
print('=' * 78)
print('EL CASO FEO: falta justamente el `nombre` -> ¿con que se identifica el archivo?')
print('=' * 78)
print('YAML (el 2do archivo no tiene con que ser nombrado):')
print(fixtures['archivo_sin_nombre.yaml'][1])
for estilo in ['indice', 'nombre', 'ambos']:
    _resultado, mensaje = probar('archivo_sin_nombre.yaml', estilo=estilo)
    print(f'  estilo={estilo:8s} -> {mensaje}')
print()
print('Lectura: con estilo="nombre" el mensaje DEGRADA a `archivo[1]`, porque no hay')
print('nombre que mostrar. Es decir: "identificar solo por nombre" no es implementable')
print('sin un fallback por indice. Ver la comparativa completa en la Celda 10.')

pd.DataFrame(filas)

=== archivo_sin_nombre.yaml ===


  ContractSchemaError: archivo[1]: falta el campo requerido 'nombre' (Field required)

=== archivo_columnas_vacias.yaml ===
  ContractSchemaError: archivo[1] 'ventas.csv': la lista de columnas no puede estar vacía

=== archivo_columnas_ausentes.yaml ===
  ContractSchemaError: archivo[1] 'ventas.csv': falta el campo requerido 'columnas' (Field required)

OK: los tres huecos de nivel archivo -> ContractSchemaError que ubica el archivo.

EL CASO FEO: falta justamente el `nombre` -> ¿con que se identifica el archivo?
YAML (el 2do archivo no tiene con que ser nombrado):

contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}

  estilo=indice   -> archivo[1]: falta el campo requerido 'nombre' (Field required)
  estilo=nombre   -> archivo[1]: falta el campo requerido 'nombre' (Field required)
  estilo=ambos    -> ar

,fixture,resultado,mensaje
0,archivo_sin_nombre.yaml,ContractSchemaError,archivo[1]: falta el campo requerido 'nombre' (Field required)
1,archivo_columnas_vacias.yaml,ContractSchemaError,archivo[1] 'ventas.csv': la lista de columnas no puede estar vacía
2,archivo_columnas_ausentes.yaml,ContractSchemaError,archivo[1] 'ventas.csv': falta el campo requerido 'columnas' (Field required)


## Celda 8 — HU-05: dos archivos con el mismo `nombre` → error claro · y la pregunta abierta **(b)**

Traza → **HU-05**.

Primero el caso base: dos archivos `ventas.csv` → `ContractSchemaError` que menciona el `nombre` duplicado.

### 🔍 Pregunta abierta (b): cadena exacta vs normalizada
El supuesto asumido en la `definition.md` es **cadena exacta**, por consistencia con D-23e (columnas duplicadas). La tabla compara ambas politicas sobre los mismos pares de nombres, y despues se demuestra **a traves del modelo real** que `Ventas.csv` + `ventas.csv` **se aceptan** con la politica exacta y **se rechazan** con la normalizada.

**Evidencia relevante para decidir (no la decidimos aqui):** a diferencia de un nombre de columna, `nombre` designa un **archivo del sistema de ficheros**. En Windows y macOS (por defecto) `Ventas.csv` y `ventas.csv` son **el mismo archivo**; en Linux son dos. Con politica exacta, un contrato con ambos se acepta hoy y el conflicto reaparece —mas tarde y mas lejos— en `load_data`. Con politica normalizada, se rechaza aqui. Ambas opciones tienen coste; el trade-off es *consistencia con D-23e* vs *fallar temprano*.

In [8]:
print('=== Caso base (HU-05): dos archivos `ventas.csv` ===')
resultado, mensaje = probar('archivos_duplicados.yaml')
print(f'  {resultado}: {mensaje}')
assert resultado == 'ContractSchemaError'
print("  OK: rechazado y el mensaje menciona el `nombre` duplicado.")
print()

print('=' * 78)
print('PREGUNTA ABIERTA (b): cadena EXACTA (supuesto, D-23e) vs NORMALIZADA')
print('=' * 78)

pares = [
    ('ventas.csv', 'ventas.csv', 'identicos'),
    ('Ventas.csv', 'ventas.csv', 'difieren solo en mayusculas'),
    ('ventas.csv ', 'ventas.csv', 'difieren solo en un espacio final'),
    ('ventas.csv', 'clientes.csv', 'realmente distintos'),
]

comparativa = pd.DataFrame(
    [
        {
            'nombre_1': repr(a),
            'nombre_2': repr(b),
            'caso': caso,
            'A) exacta -> ': 'RECHAZA' if _duplicados([a, b], False) else 'acepta',
            'B) normalizada -> ': 'RECHAZA' if _duplicados([a, b], True) else 'acepta',
            'mismo fichero en Windows/macOS?': 'SI' if a.strip().lower() == b.strip().lower() else 'no',
        }
        for a, b, caso in pares
    ]
)
print(comparativa.to_string(index=False))
print()

print('--- Demostracion a traves del modelo real, con el fixture Ventas.csv + ventas.csv ---')
for normalizado in [False, True]:
    MODO_DUPLICADOS_NORMALIZADO = normalizado
    resultado, mensaje = probar('archivos_duplicados_casing.yaml')
    etiqueta = 'NORMALIZADA' if normalizado else 'EXACTA     '
    print(f'  politica {etiqueta} -> {resultado}: {mensaje}')
MODO_DUPLICADOS_NORMALIZADO = False   # se restaura el supuesto vigente
print()
print('Trade-off a decidir por el humano:')
print('  A) EXACTA     : consistente con D-23e (columnas). `Ventas.csv` y `ventas.csv`')
print('                  se aceptan como dos archivos distintos, aunque en Windows/macOS')
print('                  sean EL MISMO fichero; la colision reaparece luego, en load_data.')
print('  B) NORMALIZADA: falla temprano, aqui. Rompe la simetria con la regla de columnas')
print('                  y anade una regla que mantener. Ojo: `nombre` es un fichero, no')
print('                  una columna, y esa asimetria podria estar justificada.')
comparativa

=== Caso base (HU-05): dos archivos `ventas.csv` ===
  ContractSchemaError: nombre de archivo duplicado: 'ventas.csv'
  OK: rechazado y el mensaje menciona el `nombre` duplicado.

PREGUNTA ABIERTA (b): cadena EXACTA (supuesto, D-23e) vs NORMALIZADA
     nombre_1       nombre_2                              caso A) exacta ->  B) normalizada ->  mismo fichero en Windows/macOS?
 'ventas.csv'   'ventas.csv'                         identicos       RECHAZA            RECHAZA                              SI
 'Ventas.csv'   'ventas.csv'       difieren solo en mayusculas        acepta            RECHAZA                              SI
'ventas.csv '   'ventas.csv' difieren solo en un espacio final        acepta            RECHAZA                              SI
 'ventas.csv' 'clientes.csv'               realmente distintos        acepta             acepta                              no

--- Demostracion a traves del modelo real, con el fixture Ventas.csv + ventas.csv ---
  politica EXACTA      -

,nombre_1,nombre_2,caso,A) exacta ->,B) normalizada ->,mismo fichero en Windows/macOS?
0,'ventas.csv','ventas.csv',identicos,RECHAZA,RECHAZA,SI
1,'Ventas.csv','ventas.csv',difieren solo en mayusculas,acepta,RECHAZA,SI
2,'ventas.csv ','ventas.csv',difieren solo en un espacio final,acepta,RECHAZA,SI
3,'ventas.csv','clientes.csv',realmente distintos,acepta,acepta,no


## Celda 9 — HU-06: validaciones de columna **dentro de cada archivo** · HOY vs PROPUESTA

Traza → **HU-06**. Las tres reglas vigentes (campo requerido faltante, `tipo` fuera del enum de 6, columna duplicada por cadena exacta) siguen aplicando, pero **dentro** de cada archivo y con el mensaje **localizando el archivo**.

### 🔍 El contraste que motiva la HU
- **HOY**, un YAML multi-archivo ni siquiera llega a las columnas: `contract_data.columnas` no existe → el motor vigente responde **`la lista de columnas no puede estar vacia`**, un mensaje **enganoso** (el contrato tiene columnas de sobra; lo que no entiende es la forma). Es exactamente lo que reporta D-18.
- **HOY**, aun sobre la forma vieja, el mensaje dice **`columna de indice 1`**: con varios archivos ese indice es **ambiguo** (¿indice 1 de cual archivo?). Se muestra invocando el `_mensaje_esquema` vigente sobre el mismo contrato "aplanado".
- **PROPUESTA**: el mismo defecto produce un mensaje que dice **en cual archivo** y **en cual columna**.

In [9]:
casos_hu06 = [
    'columna_campo_faltante.yaml',
    'columna_tipo_invalido.yaml',
    'columnas_duplicadas.yaml',
]

print('=== PROPUESTA: las 3 reglas vigentes, aplicadas DENTRO de ventas.csv ===')
filas = []
for caso in casos_hu06:
    resultado, mensaje = probar(caso)
    filas.append({'fixture': caso, 'resultado': resultado, 'mensaje': mensaje})
    print(f'  {caso:30s} -> {mensaje}')
assert all(f['resultado'] == 'ContractSchemaError' for f in filas)
print()
print('  OK: cada mensaje dice EN CUAL archivo y EN CUAL columna esta el defecto.')
print()

# Matiz importante: un mismo nombre de columna en DOS archivos distintos es legal
# (`cliente_id` esta en clientes.csv y en ventas.csv). El duplicado se juzga
# DENTRO de cada archivo, no globalmente.
contrato_ok = load_contract_propuesto(FIXTURES / 'multi_valido.yaml')
todas = [(a.nombre, c.nombre) for a in contrato_ok.archivos for c in a.columnas]
repetidas_entre_archivos = sorted({c for _a, c in todas if [x for _y, x in todas].count(c) > 1})
print('Columnas repetidas ENTRE archivos distintos (legal):', repetidas_entre_archivos)
print('  -> multi_valido.yaml carga sin error:', len(contrato_ok.archivos), 'archivos.')
print('  -> el alcance del duplicado es POR ARCHIVO, no global. La spec debe decirlo.')
print()

print('=' * 78)
print('CONTRASTE CON EL CODIGO VIGENTE (por que hace falta esta HU)')
print('=' * 78)

# (1) HOY, un YAML multi-archivo ni siquiera llega a las columnas
print('(1) Motor VIGENTE contra un contrato multi-archivo (columna_tipo_invalido.yaml):')
try:
    load_contract_vigente(FIXTURES / 'columna_tipo_invalido.yaml')
    print('    OK (inesperado)')
except ContractSchemaError as exc:
    print(f'    ContractSchemaError: {exc}')
    print('    ^ ENGANOSO: el contrato tiene 6 columnas; lo que el motor no entiende es')
    print('      la FORMA (busca contract_data.columnas y encuentra contract_data.archivos).')
    print('      Es exactamente lo que reporta D-18.')
print()

# (2) HOY, incluso sobre la forma vieja, el indice de columna es ambiguo
fixtures['aplanado_esquema_viejo.yaml'] = (
    'HU-06 contraste: mismas columnas aplanadas en la forma VIEJA',
    '''
contract_data:
  columnas:
    - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
    - {nombre: monto, tipo: numero_magico, nulable: true, llave: false}
''',
)
(FIXTURES / 'aplanado_esquema_viejo.yaml').write_text(
    fixtures['aplanado_esquema_viejo.yaml'][1], encoding='utf-8'
)

print('(2) Motor VIGENTE contra la forma vieja (aplanado_esquema_viejo.yaml):')
try:
    load_contract_vigente(FIXTURES / 'aplanado_esquema_viejo.yaml')
    print('    OK (inesperado)')
except ContractSchemaError as exc:
    print(f'    ContractSchemaError: {exc}')
    print('    ^ AMBIGUO con varios archivos: "columna de indice 1" ... ¿de cual archivo?')
print()

print('(3) PROPUESTA sobre el mismo defecto, ya en forma multi-archivo:')
_resultado, mensaje = probar('columna_tipo_invalido.yaml')
print(f'    ContractSchemaError: {mensaje}')
print('    ^ localiza archivo + columna + campo + valor invalido + los 6 permitidos.')

pd.DataFrame(filas)

=== PROPUESTA: las 3 reglas vigentes, aplicadas DENTRO de ventas.csv ===


  columna_campo_faltante.yaml    -> archivo[1] 'ventas.csv', columna de índice 1: falta el campo requerido 'tipo' (Field required)
  columna_tipo_invalido.yaml     -> archivo[1] 'ventas.csv', columna de índice 1, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', 'datetime' or 'boolean')
  columnas_duplicadas.yaml       -> archivo[1] 'ventas.csv': nombre de columna duplicado: 'venta_id'

  OK: cada mensaje dice EN CUAL archivo y EN CUAL columna esta el defecto.

Columnas repetidas ENTRE archivos distintos (legal): ['cliente_id']
  -> multi_valido.yaml carga sin error: 2 archivos.
  -> el alcance del duplicado es POR ARCHIVO, no global. La spec debe decirlo.

CONTRASTE CON EL CODIGO VIGENTE (por que hace falta esta HU)
(1) Motor VIGENTE contra un contrato multi-archivo (columna_tipo_invalido.yaml):
    ContractSchemaError: la lista de columnas no puede estar vacía
    ^ ENGANOSO: el contrato tiene 6 columnas; lo que el motor no entiende 

,fixture,resultado,mensaje
0,columna_campo_faltante.yaml,ContractSchemaError,"archivo[1] 'ventas.csv', columna de índice 1: falta el campo requerido 'tipo' (Field required)"
1,columna_tipo_invalido.yaml,ContractSchemaError,"archivo[1] 'ventas.csv', columna de índice 1, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', ..."
2,columnas_duplicadas.yaml,ContractSchemaError,archivo[1] 'ventas.csv': nombre de columna duplicado: 'venta_id'


## Celda 10 — 🔍 Pregunta abierta **(a)**: como identificar el archivo en el mensaje

Traza → **HU-04** + **HU-06** (ambas exigen "que el archivo sea identificable", sin fijar el mecanismo).

Misma bateria de contratos rotos, mismo motor, **tres politicas de localizacion**:

| Opcion | Ejemplo | Que pasa si falta el `nombre` | Que pasa si hay `nombre` duplicado |
|---|---|---|---|
| **A. Solo indice** | `archivo[1]` | funciona (siempre hay indice) | funciona (desambigua) |
| **B. Solo nombre** | `archivo 'ventas.csv'` | **no hay nombre** → hay que degradar a indice igual | **ambiguo** (dos archivos, mismo nombre) |
| **C. Ambos** | `archivo[1] 'ventas.csv'` | degrada a `archivo[1]` | desambigua |

La tabla de abajo es la **evidencia ejecutada** de esas tres columnas: el mensaje literal que recibiria el humano en cada caso.

In [10]:
# Fixture extra: dos archivos con el MISMO nombre y, ademas, una columna rota en
# el segundo. Sirve para ver si "solo nombre" desambigua cuando hay homonimos.
fixtures['duplicado_y_columna_rota.yaml'] = (
    '(a) dos ventas.csv y una columna rota en el segundo',
    '''
contract_data:
  archivos:
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas:
        - {nombre: monto, tipo: numero_magico, nulable: true, llave: false}
''',
)
(FIXTURES / 'duplicado_y_columna_rota.yaml').write_text(
    fixtures['duplicado_y_columna_rota.yaml'][1], encoding='utf-8'
)

casos_a = [
    ('archivo_sin_nombre.yaml', 'falta el `nombre` del 2do archivo'),
    ('archivo_columnas_vacias.yaml', 'ventas.csv con columnas: []'),
    ('archivo_columnas_ausentes.yaml', 'ventas.csv sin la clave columnas'),
    ('columna_tipo_invalido.yaml', 'tipo invalido en una columna de ventas.csv'),
    ('columnas_duplicadas.yaml', 'columna duplicada dentro de ventas.csv'),
    ('duplicado_y_columna_rota.yaml', 'dos ventas.csv + columna rota en el 2do'),
    ('archivos_duplicados.yaml', 'nombre de archivo duplicado (error de nivel raiz)'),
]

comparativa_a = pd.DataFrame(
    [
        {
            'caso': descripcion,
            'A) solo indice': probar(caso, estilo='indice')[1],
            'B) solo nombre': probar(caso, estilo='nombre')[1],
            'C) ambos': probar(caso, estilo='ambos')[1],
        }
        for caso, descripcion in casos_a
    ]
)

for fila in comparativa_a.to_dict('records'):
    print('=' * 78)
    print('CASO:', fila['caso'])
    print('  A) solo indice -> ', fila['A) solo indice'])
    print('  B) solo nombre -> ', fila['B) solo nombre'])
    print('  C) ambos       -> ', fila['C) ambos'])
print('=' * 78)
print()
print('Lectura de la evidencia (la decision es del humano):')
print("  * Caso 'falta el nombre': B degrada a archivo[1] -> B no existe sin fallback.")
print("  * Caso 'dos ventas.csv + columna rota': B nombra el archivo pero NO dice cual de")
print('    los dos homonimos -> B es ambiguo justo cuando mas se necesita precision.')
print('  * A siempre funciona, pero obliga a contar posiciones en el YAML a mano.')
print('  * C = indice (siempre existe, desambigua) + nombre (lo que el humano reconoce).')
print()
print("  Ojo: el error de 'nombre de archivo duplicado' es de NIVEL RAIZ (loc == ('archivos',))")
print('  y no lleva localizador de archivo en ningun estilo; su mensaje ya nombra el duplicado.')

comparativa_a

CASO: falta el `nombre` del 2do archivo
  A) solo indice ->  archivo[1]: falta el campo requerido 'nombre' (Field required)
  B) solo nombre ->  archivo[1]: falta el campo requerido 'nombre' (Field required)
  C) ambos       ->  archivo[1]: falta el campo requerido 'nombre' (Field required)
CASO: ventas.csv con columnas: []
  A) solo indice ->  archivo[1]: la lista de columnas no puede estar vacía
  B) solo nombre ->  archivo 'ventas.csv': la lista de columnas no puede estar vacía
  C) ambos       ->  archivo[1] 'ventas.csv': la lista de columnas no puede estar vacía
CASO: ventas.csv sin la clave columnas
  A) solo indice ->  archivo[1]: falta el campo requerido 'columnas' (Field required)
  B) solo nombre ->  archivo 'ventas.csv': falta el campo requerido 'columnas' (Field required)
  C) ambos       ->  archivo[1] 'ventas.csv': falta el campo requerido 'columnas' (Field required)
CASO: tipo invalido en una columna de ventas.csv
  A) solo indice ->  archivo[1], columna de índice 1, cam

,caso,A) solo indice,B) solo nombre,C) ambos
0,falta el `nombre` del 2do archivo,archivo[1]: falta el campo requerido 'nombre' (Field required),archivo[1]: falta el campo requerido 'nombre' (Field required),archivo[1]: falta el campo requerido 'nombre' (Field required)
1,ventas.csv con columnas: [],archivo[1]: la lista de columnas no puede estar vacía,archivo 'ventas.csv': la lista de columnas no puede estar vacía,archivo[1] 'ventas.csv': la lista de columnas no puede estar vacía
2,ventas.csv sin la clave columnas,archivo[1]: falta el campo requerido 'columnas' (Field required),archivo 'ventas.csv': falta el campo requerido 'columnas' (Field required),archivo[1] 'ventas.csv': falta el campo requerido 'columnas' (Field required)
3,tipo invalido en una columna de ventas.csv,"archivo[1], columna de índice 1, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', 'datetime' or...","archivo 'ventas.csv', columna de índice 1, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', 'da...","archivo[1] 'ventas.csv', columna de índice 1, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', ..."
4,columna duplicada dentro de ventas.csv,archivo[1]: nombre de columna duplicado: 'venta_id',archivo 'ventas.csv': nombre de columna duplicado: 'venta_id',archivo[1] 'ventas.csv': nombre de columna duplicado: 'venta_id'
5,dos ventas.csv + columna rota en el 2do,"archivo[1], columna de índice 0, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', 'datetime' or...","archivo 'ventas.csv', columna de índice 0, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', 'da...","archivo[1] 'ventas.csv', columna de índice 0, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', ..."
6,nombre de archivo duplicado (error de nivel raiz),nombre de archivo duplicado: 'ventas.csv',nombre de archivo duplicado: 'ventas.csv',nombre de archivo duplicado: 'ventas.csv'


## Celda 11 — HU-07: YAML roto → `ContractParseError`, distinguible del error de esquema

Traza → **HU-07**. El cambio a multi-archivo **no toca** esta distincion: el parseo falla antes de que el esquema entre en juego. Se clasifica por **tipo de excepcion** un YAML con sintaxis rota frente a uno bien parseado pero mal formado.

In [11]:
clasificacion = {}
for caso in ['yaml_roto.yaml', 'columna_tipo_invalido.yaml', 'multi_valido.yaml']:
    resultado, mensaje = probar(caso)
    clasificacion[caso] = (resultado, mensaje)

print('Clasificacion por TIPO de excepcion (parseo vs esquema vs OK):')
print()
for caso, (resultado, mensaje) in clasificacion.items():
    print(f'  {caso:28s} -> {resultado}')
    print(f'  {"":28s}    {mensaje}')
    print()

assert clasificacion['yaml_roto.yaml'][0] == 'ContractParseError'
assert clasificacion['columna_tipo_invalido.yaml'][0] == 'ContractSchemaError'
assert clasificacion['multi_valido.yaml'][0] == 'OK -> Contract'
print('OK: la sintaxis rota falla ANTES del esquema y se distingue por tipo de excepcion.')
print('    El paso a multi-archivo no altera esta distincion: el parseo es previo a la forma.')

Clasificacion por TIPO de excepcion (parseo vs esquema vs OK):

  yaml_roto.yaml               -> ContractParseError
                                  el archivo YAML del contrato es sintácticamente inválido: while parsing a block collection

  columna_tipo_invalido.yaml   -> ContractSchemaError
                                  archivo[1] 'ventas.csv', columna de índice 1, campo 'tipo': valor 'numero_magico' inválido (Input should be 'string', 'integer', 'float', 'date', 'datetime' or 'boolean')

  multi_valido.yaml            -> OK -> Contract
                                  2 archivo(s): clientes.csv, ventas.csv

OK: la sintaxis rota falla ANTES del esquema y se distingue por tipo de excepcion.
    El paso a multi-archivo no altera esta distincion: el parseo es previo a la forma.


## Celda 12 — HU-08 (bug **T-56**): `contract_data:` presente pero nulo

Traza → **HU-08**.

El bug vive en la **misma linea** que esta feature reescribe (`contract.py:115`):

```python
columnas = (crudo or {}).get('contract_data', {}).get('columnas', [])
```

Si `contract_data:` existe con valor nulo, `crudo` es `{'contract_data': None}`: el primer `.get` devuelve **`None`** (no el default `{}`, porque **la clave si existe**) y el segundo `.get` revienta con **`AttributeError`** — un detalle de implementacion filtrado al usuario. Es el caso de la plantilla real recien copiada y aun vacia.

Abajo se demuestra **contra el codigo vigente** (se captura `Exception` a proposito, para probar que hoy **no** es un `ContractSchemaError`) y luego el comportamiento de la propuesta. Se incluyen variantes hermanas: YAML totalmente vacio, y `contract_data` con un valor que no es un mapa.

In [12]:
print('Contenido del fixture contract_data_nulo.yaml (la plantilla recien copiada):')
print(repr(fixtures['contract_data_nulo.yaml'][1]))
print('Lo que ve el parser  :', repr(yaml.safe_load(fixtures['contract_data_nulo.yaml'][1])))
print('  -> la clave EXISTE, su valor es None: por eso el default {} del .get() no se usa.')
print()

print('=' * 78)
print('HOY (codigo vigente en main) — se captura Exception a proposito')
print('=' * 78)
casos_t56 = ['contract_data_nulo.yaml', 'yaml_totalmente_vacio.yaml', 'contract_data_no_mapa.yaml']
filas_hoy = []
for caso in casos_t56:
    try:
        load_contract_vigente(FIXTURES / caso)
        etiqueta, detalle = 'OK -> Contract', '(inesperado)'
    except (ContractParseError, ContractSchemaError) as exc:
        etiqueta, detalle = type(exc).__name__, str(exc)
    except Exception as exc:                     # noqa: BLE001  (es el bug)
        etiqueta, detalle = f'!! {type(exc).__name__} NO CONTROLADA', str(exc)
    filas_hoy.append({'fixture': caso, 'HOY': etiqueta, 'mensaje HOY': detalle})
    print(f'  {caso:28s} -> {etiqueta}')
    print(f'  {"":28s}    {detalle}')
print()
print('  ^ T-56 confirmado: `AttributeError` filtra un detalle de implementacion')
print('    (`.get` sobre None) a quien solo copio la plantilla y aun no la lleno.')
print()

print('=' * 78)
print('PROPUESTA (extraccion defensiva de la raiz)')
print('=' * 78)
filas = []
for caso, fila_hoy in zip(casos_t56, filas_hoy):
    resultado, mensaje = probar(caso)
    print(f'  {caso:28s} -> {resultado}')
    print(f'  {"":28s}    {mensaje}')
    filas.append({**fila_hoy, 'PROPUESTA': resultado, 'mensaje PROPUESTA': mensaje})

assert all(f['PROPUESTA'] == 'ContractSchemaError' for f in filas)
print()
print('OK: los tres casos dan ContractSchemaError claro; ningun AttributeError se propaga.')
pd.DataFrame(filas)

Contenido del fixture contract_data_nulo.yaml (la plantilla recien copiada):
'\ncontract_data:\n'
Lo que ve el parser  : {'contract_data': None}
  -> la clave EXISTE, su valor es None: por eso el default {} del .get() no se usa.

HOY (codigo vigente en main) — se captura Exception a proposito
  contract_data_nulo.yaml      -> !! AttributeError NO CONTROLADA
                                  'NoneType' object has no attribute 'get'
  yaml_totalmente_vacio.yaml   -> ContractSchemaError
                                  la lista de columnas no puede estar vacía
  contract_data_no_mapa.yaml   -> !! AttributeError NO CONTROLADA
                                  'str' object has no attribute 'get'

  ^ T-56 confirmado: `AttributeError` filtra un detalle de implementacion
    (`.get` sobre None) a quien solo copio la plantilla y aun no la lleno.

PROPUESTA (extraccion defensiva de la raiz)
  contract_data_nulo.yaml      -> ContractSchemaError
                                  la clave raíz 'c

,fixture,HOY,mensaje HOY,PROPUESTA,mensaje PROPUESTA
0,contract_data_nulo.yaml,!! AttributeError NO CONTROLADA,'NoneType' object has no attribute 'get',ContractSchemaError,la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: None)
1,yaml_totalmente_vacio.yaml,ContractSchemaError,la lista de columnas no puede estar vacía,ContractSchemaError,la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: None)
2,contract_data_no_mapa.yaml,!! AttributeError NO CONTROLADA,'str' object has no attribute 'get',ContractSchemaError,la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: 'pendiente_de_completar')


## Celda 13 — HU-09 (**T-58**): como quedaria `600_template/contract_data.yaml`

Traza → **HU-09**. Se **propone** el contenido de la plantilla en esquema multi-archivo, se materializa en el directorio **temporal** y se comprueba que **carga sin error** con el motor propuesto.

> El archivo real `600_template/contract_data.yaml` **no se modifica aqui**: es trabajo del bucle TDD (paso 10) tras la spec. Esta celda solo hace visible la forma propuesta para que el humano la apruebe o la corrija en el gate.

Cambios respecto de la plantilla vigente:
- `contract_data.columnas:` → `contract_data.archivos:` con `- nombre: <archivo>` + `columnas:`.
- Se documentan las reglas nuevas de nivel archivo (al menos un archivo; `nombre` obligatorio; sin `nombre` duplicado).
- Se corrige el comando de verificacion (la plantilla vigente apunta a `610_features/config_contract/cargar_contrato.py`).

In [13]:
plantilla_propuesta = '''# =============================================================================
# contract_data.yaml — PLANTILLA (guía para el humano)
# =============================================================================
#
# QUÉ ES: el contrato de datos de un cliente. Declara qué archivos entrega el
# cliente y, para cada uno, qué columnas debe tener, de qué tipo es cada una,
# si admite nulos y si es llave.
#
# CÓMO SE USA:
#   1. Copia este archivo a  clients/<TU_CLIENTE>/input/contract_data.yaml
#   2. Borra los archivos y columnas de ejemplo y escribe los tuyos.
#   3. Verifica que el motor lo entiende (carga sin error).
#
# REGLAS (si no se cumplen, la carga falla con un error que dice por qué):
#   - Debe haber AL MENOS UN archivo bajo `archivos:`. La lista no puede quedar
#     vacía.
#   - Cada archivo necesita `nombre` y una lista `columnas` con AL MENOS UNA
#     columna.
#   - No se permiten dos archivos con el mismo `nombre`.
#   - Cada columna necesita los CUATRO campos: nombre, tipo, nulable, llave.
#   - `tipo` solo admite estos seis valores, escritos igual:
#         string · integer · float · date · datetime · boolean
#   - `nulable` y `llave` son true/false.
#   - No se permiten dos columnas con el mismo `nombre` DENTRO DEL MISMO archivo
#     (comparación exacta: `test_id` y `Test_ID` se consideran distintos). Dos
#     archivos distintos SÍ pueden tener columnas que se llamen igual.
#   - El orden de los archivos y el de las columnas se conserva.
#
# QUÉ SIGNIFICA CADA CAMPO:
#   nombre (archivo) → el nombre del archivo de datos, tal cual lo entrega el
#                      cliente (p. ej. ventas.csv). Es una etiqueta declarativa.
#   nombre (columna) → el nombre de la columna, tal cual aparece en el archivo.
#   tipo             → el tipo de dato esperado (uno de los seis de arriba).
#   nulable          → true si la columna puede venir vacía; false si siempre
#                      debe traer valor.
#   llave            → true si la columna identifica la fila (id, código).
#
# OJO: no borres la línea `contract_data:` ni la línea `archivos:`. Si dejas
# `contract_data:` sin nada debajo, el archivo no es un contrato válido.
# =============================================================================

contract_data:
  archivos:
    # --- Ejemplos: bórralos y pon tus archivos reales ------------------------
    - nombre: clientes.csv
      columnas:
        - nombre: cliente_id
          tipo: integer
          nulable: false
          llave: true

        - nombre: correo
          tipo: string
          nulable: true
          llave: false

        - nombre: fecha_alta
          tipo: date
          nulable: false
          llave: false

    - nombre: ventas.csv
      columnas:
        - nombre: venta_id
          tipo: integer
          nulable: false
          llave: true

        - nombre: cliente_id
          tipo: integer
          nulable: false
          llave: false

        - nombre: monto
          tipo: float
          nulable: true
          llave: false

        - nombre: vendida_en
          tipo: datetime
          nulable: false
          llave: false

        - nombre: anulada
          tipo: boolean
          nulable: false
          llave: false
'''

# Se materializa en el TEMPORAL del spike; el archivo real de 600_template/ NO se toca.
fixtures['plantilla_propuesta.yaml'] = ('HU-09/T-58 plantilla multi-archivo propuesta', plantilla_propuesta)
(FIXTURES / 'plantilla_propuesta.yaml').write_text(plantilla_propuesta, encoding='utf-8')

print(plantilla_propuesta)
print('=' * 78)
print('¿Carga sin error con el motor propuesto? (criterio de HU-09)')
plantilla = load_contract_propuesto(FIXTURES / 'plantilla_propuesta.yaml')
print('  OK ->', type(plantilla).__name__, 'con', len(plantilla.archivos), 'archivos:',
      [a.nombre for a in plantilla.archivos])
assert len(plantilla.archivos) >= 1
assert all(a.nombre and a.columnas for a in plantilla.archivos)
print()

ruta_plantilla_real = RAIZ / '600_template' / 'contract_data.yaml'
print('Estado del archivo REAL (no se modifica en este spike):')
print(' ', ruta_plantilla_real)
try:
    load_contract_propuesto(ruta_plantilla_real)
    print('  carga con el motor propuesto: OK (inesperado, sigue en esquema viejo)')
except ContractSchemaError as exc:
    print(f'  carga con el motor propuesto: ContractSchemaError -> {exc}')
    print('  ^ correcto: la plantilla vigente esta en el esquema viejo. Actualizarla es')
    print('    trabajo del bucle TDD (paso 10), no de este spike.')

pd.DataFrame(
    [
        {'archivo': a.nombre, 'columnas': len(a.columnas),
         'nombres': ', '.join(c.nombre for c in a.columnas)}
        for a in plantilla.archivos
    ]
)

# =============================================================================
# contract_data.yaml — PLANTILLA (guía para el humano)
# =============================================================================
#
# QUÉ ES: el contrato de datos de un cliente. Declara qué archivos entrega el
# cliente y, para cada uno, qué columnas debe tener, de qué tipo es cada una,
# si admite nulos y si es llave.
#
# CÓMO SE USA:
#   1. Copia este archivo a  clients/<TU_CLIENTE>/input/contract_data.yaml
#   2. Borra los archivos y columnas de ejemplo y escribe los tuyos.
#   3. Verifica que el motor lo entiende (carga sin error).
#
# REGLAS (si no se cumplen, la carga falla con un error que dice por qué):
#   - Debe haber AL MENOS UN archivo bajo `archivos:`. La lista no puede quedar
#     vacía.
#   - Cada archivo necesita `nombre` y una lista `columnas` con AL MENOS UNA
#     columna.
#   - No se permiten dos archivos con el mismo `nombre`.
#   - Cada columna necesita los CUATRO campos: nombre,

,archivo,columnas,nombres
0,clientes.csv,3,"cliente_id, correo, fecha_alta"
1,ventas.csv,5,"venta_id, cliente_id, monto, vendida_en, anulada"


## Celda 14 — HU-10: frontera con `load_data` (no se toca ningun CSV de bronze)

Traza → **HU-10**. Instrumentamos `builtins.open` durante una carga completa del contrato multi-archivo y verificamos que **la unica ruta abierta es el propio `contract_data.yaml`**: ni `clientes.csv`, ni `ventas.csv`, ni nada bajo `data/bronze/`, pese a que el contrato **nombra** esos archivos. El `nombre` es una cadena declarativa (D-21).

In [14]:
import builtins

rutas_abiertas: list[str] = []
_open_original = builtins.open


def _open_espia(archivo, *args, **kwargs):
    rutas_abiertas.append(str(archivo))
    return _open_original(archivo, *args, **kwargs)


# OJO: dentro de un kernel de Jupyter, parchear `builtins.open` NO intercepta el
# `open()` que llama una función definida en una celda: el espáa queda mudo y los
# asserts pasarán en vacáo. Se sombrea `open` como GLOBAL del notebook, que gana
# en la b\u00fasqueda de nombres. (En pytest, donde vive el test real de la frontera,
# `monkeypatch.setattr(builtins, "open", ...)` sá funciona: es un proceso normal.)
open = _open_espia
try:
    espiado = load_contract_propuesto(FIXTURES / 'multi_valido.yaml')
finally:
    del open

print('El contrato NOMBRA estos archivos:', [a.nombre for a in espiado.archivos])
print()
print('Rutas realmente abiertas durante load_contract_propuesto:')
for ruta in rutas_abiertas:
    print('  -', ruta)
print()

toca_bronze = any('bronze' in r.replace(chr(92), '/') for r in rutas_abiertas)
toca_clients = any('clients/' in r.replace(chr(92), '/') for r in rutas_abiertas)
toca_csv = any(r.lower().endswith('.csv') for r in rutas_abiertas)
solo_el_yaml = len(rutas_abiertas) == 1 and rutas_abiertas[0].endswith('multi_valido.yaml')

print('Rutas observadas por el espia       ->', len(rutas_abiertas))
print('¿Abrio algo bajo data/bronze/?      ->', toca_bronze)
print('¿Abrio algo bajo clients/?          ->', toca_clients)
print('¿Abrio algun .csv (los nombrados)?  ->', toca_csv)
print('¿Abrio SOLO el contract_data.yaml?  ->', solo_el_yaml)

# Guarda de no-vacuidad: si el espia no observo NADA, la ausencia de bronze no
# prueba nada (una lista vacia no contiene nada). La evidencia debe ser positiva.
assert rutas_abiertas, 'el espia no observo ninguna apertura: evidencia vacua'
assert solo_el_yaml, 'se esperaba exactamente una apertura: el propio contract_data.yaml'
assert not toca_bronze and not toca_clients and not toca_csv
print()
print('OK: `nombre` es una cadena declarativa. El motor no la resuelve, no la busca y')
print('    no la abre. Emparejar el nombre con el fichero ingerido es de load_data (D-21).')


El contrato NOMBRA estos archivos: ['clientes.csv', 'ventas.csv']

Rutas realmente abiertas durante load_contract_propuesto:
  - C:\Users\USUARIO\AppData\Local\Temp\contract_multifile_spike_na5n32l9\multi_valido.yaml

Rutas observadas por el espia       -> 1
¿Abrio algo bajo data/bronze/?      -> False
¿Abrio algo bajo clients/?          -> False
¿Abrio algun .csv (los nombrados)?  -> False
¿Abrio SOLO el contract_data.yaml?  -> True

OK: `nombre` es una cadena declarativa. El motor no la resuelve, no la busca y
    no la abre. Emparejar el nombre con el fichero ingerido es de load_data (D-21).


## Celda 15 — HU-11: como se migran los tests de `config_contract`

Traza → **HU-11**. La migracion es **mecanica**: un nivel mas de indexado. Se muestra la tabla de equivalencias y se ejecuta la version nueva de cada assert sobre el contrato de un solo archivo, para evidenciar que **no se pierde cobertura de comportamiento**.

> Punto que la spec debe zanjar: **no** se propone ningun atajo de compatibilidad (p. ej. una propiedad `Contract.columnas` que aplane los archivos). Seria una segunda forma de leer lo mismo y contradice HU-02 ("un archivo es el caso general, no una rama especial"). Se migran los tests y punto.

In [15]:
migracion = pd.DataFrame(
    [
        {
            'comportamiento verificado hoy': 'count de columnas',
            'assert VIEJO': 'len(contrato.columnas) == 3',
            'assert NUEVO': 'len(contrato.archivos[0].columnas) == 3',
        },
        {
            'comportamiento verificado hoy': 'orden declarado',
            'assert VIEJO': "[c.nombre for c in contrato.columnas] == [...]",
            'assert NUEVO': "[c.nombre for c in contrato.archivos[0].columnas] == [...]",
        },
        {
            'comportamiento verificado hoy': 'fidelidad de campos',
            'assert VIEJO': "contrato.columnas[0].tipo is TipoDato.INTEGER",
            'assert NUEVO': "contrato.archivos[0].columnas[0].tipo is TipoDato.INTEGER",
        },
        {
            'comportamiento verificado hoy': 'campo requerido faltante',
            'assert VIEJO': 'ContractSchemaError sobre contract_data.columnas',
            'assert NUEVO': 'ContractSchemaError sobre archivos[i].columnas (+ archivo en el mensaje)',
        },
        {
            'comportamiento verificado hoy': 'tipo fuera del enum de 6',
            'assert VIEJO': 'ContractSchemaError, mensaje con los 6 valores',
            'assert NUEVO': 'igual, con el archivo localizado',
        },
        {
            'comportamiento verificado hoy': 'columna duplicada (cadena exacta)',
            'assert VIEJO': 'ContractSchemaError en Contract',
            'assert NUEVO': 'ContractSchemaError en ArchivoContrato (alcance: por archivo)',
        },
        {
            'comportamiento verificado hoy': 'columnas vacia/ausente',
            'assert VIEJO': 'ContractSchemaError en Contract.columnas',
            'assert NUEVO': 'ContractSchemaError en archivos[i].columnas',
        },
        {
            'comportamiento verificado hoy': 'YAML roto',
            'assert VIEJO': 'ContractParseError',
            'assert NUEVO': 'ContractParseError (sin cambios)',
        },
        {
            'comportamiento verificado hoy': 'frontera de lectura (no bronze)',
            'assert VIEJO': 'solo abre el YAML',
            'assert NUEVO': 'solo abre el YAML (sin cambios)',
        },
        {
            'comportamiento verificado hoy': 'invocable sin CLI',
            'assert VIEJO': 'load_contract(path) directo',
            'assert NUEVO': 'load_contract(path) directo (misma firma)',
        },
    ]
)

# Los asserts NUEVOS, ejecutados de verdad sobre el contrato de un solo archivo
c = load_contract_propuesto(FIXTURES / 'un_archivo.yaml')
assert len(c.archivos[0].columnas) == 3
assert [col.nombre for col in c.archivos[0].columnas] == ['cliente_id', 'correo', 'fecha_alta']
assert c.archivos[0].columnas[0].tipo is TipoDato.INTEGER
assert probar('columna_campo_faltante.yaml')[0] == 'ContractSchemaError'
assert probar('columna_tipo_invalido.yaml')[0] == 'ContractSchemaError'
assert probar('columnas_duplicadas.yaml')[0] == 'ContractSchemaError'
assert probar('archivo_columnas_vacias.yaml')[0] == 'ContractSchemaError'
assert probar('yaml_roto.yaml')[0] == 'ContractParseError'
print('OK: los 10 comportamientos siguen verificados bajo la forma archivos[].columnas.')
print('    La migracion es un nivel mas de indexado; no se pierde cobertura (HU-11).')
print()
print('Coste de la migracion: solo cambia la RUTA de acceso y, en los mensajes, la')
print('cadena esperada (ahora localiza el archivo). Ningun comportamiento desaparece.')
migracion

OK: los 10 comportamientos siguen verificados bajo la forma archivos[].columnas.
    La migracion es un nivel mas de indexado; no se pierde cobertura (HU-11).

Coste de la migracion: solo cambia la RUTA de acceso y, en los mensajes, la
cadena esperada (ahora localiza el archivo). Ningun comportamiento desaparece.


,comportamiento verificado hoy,assert VIEJO,assert NUEVO
0,count de columnas,len(contrato.columnas) == 3,len(contrato.archivos[0].columnas) == 3
1,orden declarado,[c.nombre for c in contrato.columnas] == [...],[c.nombre for c in contrato.archivos[0].columnas] == [...]
2,fidelidad de campos,contrato.columnas[0].tipo is TipoDato.INTEGER,contrato.archivos[0].columnas[0].tipo is TipoDato.INTEGER
3,campo requerido faltante,ContractSchemaError sobre contract_data.columnas,ContractSchemaError sobre archivos[i].columnas (+ archivo en el mensaje)
4,tipo fuera del enum de 6,"ContractSchemaError, mensaje con los 6 valores","igual, con el archivo localizado"
5,columna duplicada (cadena exacta),ContractSchemaError en Contract,ContractSchemaError en ArchivoContrato (alcance: por archivo)
6,columnas vacia/ausente,ContractSchemaError en Contract.columnas,ContractSchemaError en archivos[i].columnas
7,YAML roto,ContractParseError,ContractParseError (sin cambios)
8,frontera de lectura (no bronze),solo abre el YAML,solo abre el YAML (sin cambios)
9,invocable sin CLI,load_contract(path) directo,load_contract(path) directo (misma firma)


## Celda 16 — HU-12: inventario de fixtures (C-01)

Traza → **HU-12**. Todos los fixtures viven en el directorio temporal del spike, ninguno bajo `clients/*/data/`, y su contenido son nombres de archivo y columnas ficticios, sin PII.

In [16]:
import re

PATRONES_PII = [
    r'\b\d{6,}\b',                       # cedulas / telefonos / ids largos reales
    r'[\w.+-]+@(?!correo\.com)[\w-]+\.\w+',   # correos que no sean el ficticio de ejemplo
]


def es_ruta_de_boveda(ruta: str) -> bool:
    """True si la ruta cae bajo clients/*/data o bronze (datos reales, C-01)."""
    r = str(ruta).replace('\\', '/')
    return ('clients/' in r) or ('/data/' in r) or ('bronze' in r)


inventario = pd.DataFrame(
    [
        {
            'fixture': nombre,
            'origen': 'sintetico (escrito en este notebook)',
            'ruta': 'directorio temporal del spike',
            'bajo_clients_data': es_ruta_de_boveda(FIXTURES / nombre),
            'sospecha_pii': bool([p for p in PATRONES_PII if re.search(p, contenido)]),
            'proposito / traza': proposito,
        }
        for nombre, (proposito, contenido) in fixtures.items()
    ]
)

print('Directorio de fixtures      :', FIXTURES)
print('Total de fixtures           :', len(inventario))
print('¿Alguno bajo clients/*/data?:', bool(inventario['bajo_clients_data'].any()))
print('¿Alguno con sospecha de PII?:', bool(inventario['sospecha_pii'].any()))
assert not inventario['bajo_clients_data'].any()
assert not inventario['sospecha_pii'].any()
print()
print('OK (C-01): 100% sintetico. Los nombres declarados (clientes.csv, ventas.csv) y las')
print('columnas (cliente_id, correo, monto...) son ficticios; ademas, el contrato describe')
print('ESTRUCTURA, no contenido: ni siquiera existen filas de datos en este notebook.')
inventario

Directorio de fixtures      : C:\Users\USUARIO\AppData\Local\Temp\contract_multifile_spike_na5n32l9
Total de fixtures           : 21
¿Alguno bajo clients/*/data?: False
¿Alguno con sospecha de PII?: False

OK (C-01): 100% sintetico. Los nombres declarados (clientes.csv, ventas.csv) y las
columnas (cliente_id, correo, monto...) son ficticios; ademas, el contrato describe
ESTRUCTURA, no contenido: ni siquiera existen filas de datos en este notebook.


,fixture,origen,ruta,bajo_clients_data,sospecha_pii,proposito / traza
0,multi_valido.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-01 camino feliz multi-archivo
1,un_archivo.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-02 un solo archivo (forma NUEVA)
2,un_archivo_esquema_viejo.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-02/HU-11 mismo contrato en la forma VIEJA
3,archivos_vacia.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-03 archivos: []
4,archivos_nula.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-03 archivos: (nulo)
5,archivos_ausente_esquema_viejo.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-03 clave archivos ausente (contrato del esquema viejo)
6,archivo_sin_nombre.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-04 el 2do archivo no declara nombre (CASO FEO)
7,archivo_columnas_vacias.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-04 el 2do archivo tiene columnas: []
8,archivo_columnas_ausentes.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-04 el 2do archivo no declara columnas
9,archivos_duplicados.yaml,sintetico (escrito en este notebook),directorio temporal del spike,False,False,HU-05 dos archivos ventas.csv (cadena exacta)


## Resumen de cobertura y hallazgos para el gate (paso 5) y la spec (paso 6)

### Cobertura por historia
| HU | Demostrada en | Resultado visible |
|---|---|---|
| HU-01 | Celda 4 | tabla: 2 archivos con sus columnas, en orden |
| HU-02 | Celda 5 | `len(archivos) == 1` y columnas equivalentes a la forma vieja |
| HU-03 | Celda 6 | `archivos` vacia / nula / ausente → `ContractSchemaError` |
| HU-04 | Celda 7 | error que localiza el archivo; **caso feo** sin `nombre` |
| HU-05 | Celda 8 | `nombre` de archivo duplicado → error que lo menciona |
| HU-06 | Celda 9 | validaciones de columna por archivo; HOY (ambiguo) vs PROPUESTA |
| HU-07 | Celda 11 | `ContractParseError` vs `ContractSchemaError` distinguibles |
| HU-08 | Celda 12 | T-56: `AttributeError` hoy → `ContractSchemaError` propuesto |
| HU-09 | Celda 13 | T-58: plantilla multi-archivo propuesta, cargada sin error |
| HU-10 | Celda 14 | espia de `open`: solo el YAML; nunca los `.csv` nombrados |
| HU-11 | Celda 15 | tabla de migracion de asserts, ejecutada |
| HU-12 | Celdas 2 y 16 | inventario: fixtures 100% sinteticos, sin PII |

### Que funciono (recomendaciones para la spec)
1. **El nucleo se reutiliza intacto.** `Columna`, `TipoDato`, las excepciones y los dos validadores de columna sobreviven **sin un solo cambio de logica**: los validadores solo **cambian de casa** (de `Contract` a `ArchivoContrato`). La feature es genuinamente "un nivel mas de anidamiento".
2. **Un archivo = caso general.** `len(archivos) == 1` no necesita ninguna rama especial (Celda 5).
3. **T-56 se cierra con la extraccion defensiva de la raiz** (`contract_data` debe ser un mapa), que es la misma linea que la feature reescribe. Confirma que absorberlo aqui evita trabajo desechado.
4. **El fail-fast (D-23c) sigue funcionando** con `exc.errors()[0]`; solo cambia el desarmado de `loc`, que pasa de `('columnas', j, campo)` a `('archivos', i, 'columnas', j, campo)`.

### 🔴 Puntos abiertos que decide el humano en el gate
- **(a) Identificacion del archivo en el mensaje.** Evidencia (Celdas 7 y 10): la opcion "solo `nombre`" **no es viable sola** (cuando falta el `nombre` hay que degradar a indice igualmente, y con nombres duplicados el nombre no desambigua). Queda elegir entre **A (solo indice)** y **C (indice + nombre)**. El spike **recomienda C**: `archivo[1] 'ventas.csv'` — el indice siempre existe y desambigua; el nombre es lo que el humano reconoce al abrir el YAML. **Decide el humano.**
- **(b) Duplicado de `nombre` de archivo: exacta o normalizada.** Evidencia (Celda 8): con **exacta** (supuesto vigente, consistente con D-23e) `Ventas.csv` y `ventas.csv` **se aceptan** como dos archivos distintos, aunque en Windows/macOS sean **el mismo fichero**; con **normalizada** se rechazan aqui. El spike **no recomienda**: es un trade-off de producto (consistencia con D-23e vs fallar temprano). Opcion intermedia posible: mantener exacta y que `load_data` detecte la colision fisica. **Decide el humano.**
- **(c) ¿Mensaje especial para el esquema viejo?** (Celda 6) Un contrato en la forma vieja (`contract_data.columnas`) hoy cae en el generico "la lista de archivos no puede estar vacia", que no ayuda a quien tiene un YAML del esquema anterior. ¿Vale un mensaje dedicado del tipo *"se encontro 'columnas' en la raiz: el contrato usa el esquema anterior; ahora las columnas van dentro de `archivos[]`"*? Util para la migracion; es una regla mas que mantener. **Abierto.**
- **(d) Texto exacto y orden de los campos del mensaje.** El spike propone `archivo[i] 'nombre', columna de indice j, campo 'x': …`. La spec debe congelarlo (los tests lo van a fijar).
- **(e) Sin atajo de compatibilidad.** (Celda 15) Se propone **no** anadir `Contract.columnas` como propiedad de compatibilidad; se migran los tests. Confirmar en el gate.

### Riesgos tecnicos detectados
- **El nombre del archivo no viene en el error de Pydantic** cuando el defecto esta en una columna: hay que recuperarlo del **YAML crudo** por indice (Celda 3). Es la razon por la que el mensaje se compone con el crudo a la vista, y un detalle que la spec debe reflejar (el traductor de errores necesita **dos** entradas: la `ValidationError` y los archivos crudos).
- **Orden de disparo de los validadores:** los `field_validator` de lista (`archivos` no vacia, `archivos` sin duplicados) corren **despues** de validar cada item. Un contrato con un archivo mal formado **y** nombres duplicados reporta primero el error del item. Es coherente con el fail-fast, pero la spec no debe escribir un `CA-xx` que asuma el orden contrario.

> **Siguiente paso: gate humano (paso 5).** El humano revisa y aprueba este spike antes de pasar a `spec_writer` (paso 6). El notebook **no** es fuente de verdad del producto: la spec + el bucle TDD reescriben la logica en `app/src/`.